# Ablation 2026-09-01 — FlowMatching + Encoder + Temp only
**Run:** `abl_fm_enc_temp`  |  **W&B:** `1_Sep_2026_fm_enc_temp`  
**Model:** FlowMatchingModel (Euler)  |  **Encoder:** RRDB warm-start  |  **Fields:** `temperature` only


In [ ]:
RUN_NAME='abl_fm_enc_temp'; WANDB_RUN_NAME='1_Sep_2026_fm_enc_temp'
FLOW_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/abl_fm_enc_temp'
ENC_RUN_DIR='/trace/group/forgelab/ngng/multifield/DiffusionSR_shohom/diffusionsr/runs/ablation_20260901/enc_temp'
DATA_ROOT='/trace/group/forgelab/ngng/multifield/data_fields'
EVAL_OUT_DIR='/trace/group/forgelab/ngng/multifield/eval_results/ablation_20260901/abl_fm_enc_temp'
FIELD_NAMES=['temperature']; N_STEPS=3; DOWNSCALE_METHOD='direct'; NORMALIZE='standardize'
TIMESTEPS=1000; SCHEDULE='linear'; FM_N_STEPS=50; ENCODING=True; CONDITIONING='implicit'; DEVICE='cuda'
BATCH_INDEX=0; SAMPLE_INDEX=0; BATCH_SIZE=4; T_LIQ=1700.0; LIQ_THR=0.5
MELT_THRESHOLD=1900.0; ANALYSIS_CH=0; ANALYSIS_MAX_BATCH=None

In [ ]:
%matplotlib inline
import os, sys, time; from pathlib import Path
import numpy as np, torch, pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from scipy.ndimage import gaussian_filter as _gf
from IPython.display import display as _ipy_display
def _show(*a, **kw):
    for n in plt.get_fignums(): _ipy_display(plt.figure(n))
    plt.close('all')
plt.show = _show
if not torch.cuda.is_available() and DEVICE=='cuda': DEVICE='cpu'; print('CPU fallback')
def find_root(s=Path.cwd()):
    for p in [s,*s.parents]:
        if (p/'setup.py').exists() and (p/'diffusionsr').exists(): return p
    raise RuntimeError('no project root')
PROJECT_ROOT=find_root()
if str(PROJECT_ROOT) not in sys.path: sys.path.insert(0,str(PROJECT_ROOT))
print(f'root={PROJECT_ROOT} device={DEVICE}')

In [ ]:
from diffusionsr.datasets.dataset import SimulationXZDataset
from diffusionsr.analysis.analysis_functions import get_profile, load_encoder
from scipy.spatial import KDTree; from scipy.ndimage import binary_erosion
def as_numpy(x): return x.detach().cpu().numpy() if isinstance(x,torch.Tensor) else np.asarray(x)
def mae_rmse(p,g): p,g=np.asarray(p).ravel(),np.asarray(g).ravel(); return {'MAE':float(np.mean(np.abs(p-g))),'RMSE':float(np.sqrt(np.mean((p-g)**2)))}
def boundary_pixels(m): return np.argwhere(m&~binary_erosion(m,structure=np.ones((3,3))))
def chamfer_dist(a,b):
    ba,bb=boundary_pixels(a),boundary_pixels(b)
    if len(ba)==0 or len(bb)==0: return float('nan')
    return float((KDTree(bb).query(ba)[0].mean()+KDTree(ba).query(bb)[0].mean())/2)
def consistency_metrics(bt,bl):
    i,u=(bt&bl).sum(),(bt|bl).sum(); return (i/u if u>0 else float('nan')),float(np.mean((bt.astype(float)-bl.astype(float))**2)),chamfer_dist(bt,bl)
fn=FIELD_NAMES; has_sdf='sdfliqlabel' in fn; has_T='temperature' in fn; has_liq='liqlabel' in fn or has_sdf
def _liq_mask(p): return p[fn.index('sdfliqlabel')]<0 if has_sdf else p[fn.index('liqlabel')]>LIQ_THR
def _mk_contour(ax,p,sigma=1.5):
    if not(has_T and has_liq): return
    try: ax.contour(_gf(_liq_mask(p).T.astype(float),sigma),levels=[0.5],colors=['white'],linewidths=[1.],origin='lower',alpha=0.85)
    except: pass

In [ ]:
# ── Melt-pool physical metrics engine ─────────────────────────────────────────
from diffusionsr.analysis.meltpool import MeltPoolMetrics

VOXEL_SIZE_UM   = (10.0, 10.0)   # canonical (depth, length) order; hrmesh=10 µm
PLATE_HEIGHT_UM = 400.0           # plate surface at depth pixel 40 × 10 µm

metrics_engine = MeltPoolMetrics.from_dataset(
    test_ds,
    spatial_axes=("length", "depth"),
    voxel_size_um=VOXEL_SIZE_UM,
    plate_height_um=PLATE_HEIGHT_UM,
)
_MP_METRICS_T = ['depth_um', 'length_um', 'area_um2']
_MP_METRICS_L = ['keyhole_depth_um', 'leading_wall_angle_deg'] if has_liq else []
print(f'MeltPoolMetrics ready.  available: {metrics_engine.available_metrics}')

In [ ]:
from diffusionsr.runners.train_flow_matching import FlowMatchingModel
kw=dict(downscale_method=DOWNSCALE_METHOD,root_folder=DATA_ROOT,normalize=NORMALIZE,n_steps=N_STEPS,field_names=FIELD_NAMES)
train_ds,dev_ds,test_ds=(SimulationXZDataset(split=s,**kw) for s in ['train','dev','test'])
print(f'Fields:{train_ds.field_names} HR:{train_ds.img_shape} {train_ds.factor}x')
lr_enc=load_encoder(ENC_RUN_DIR,train_ds); print(f'Encoder: {ENC_RUN_DIR}')
model=FlowMatchingModel(results_folder=FLOW_RUN_DIR,lr_encoder_folder=ENC_RUN_DIR,
    train_dataset=train_ds,dev_dataset=dev_ds,test_dataset=test_ds,
    timesteps=TIMESTEPS,conditioning=CONDITIONING,encoding=ENCODING,schedule=SCHEDULE,device=DEVICE,enc_output=False)
model.load_saved_model(); print(f'FlowMatching loaded: {FLOW_RUN_DIR}')

In [ ]:
VIZ_CH = (N_STEPS - 1) * len(FIELD_NAMES)   # current-timestep temperature channel

# Find the best visualization batch: max melt pool area from middle 50% of test set
_nb = max(1, len(test_ds) // BATCH_SIZE)
_step = max(1, _nb // 100)
_start, _end = _nb // 4, 3 * _nb // 4
loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
batch = None; _best_area = -1
for i, _b in enumerate(loader):
    if i > _end: break
    if i >= _start and (i - _start) % _step == 0:
        _, _hr, _, _ = _b[:4]
        _a = float((test_ds.unscale_data(_hr.numpy()[0], input_type='hr')[VIZ_CH] > MELT_THRESHOLD).sum())
        if _a > _best_area: _best_area, batch = _a, _b
if batch is None:
    for i, batch in enumerate(DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)):
        if i == _nb // 2: break

_, hr_s, lr_s, ul_s = [x[SAMPLE_INDEX:SAMPLE_INDEX+1] for x in batch[:4]]
x_e = model.compute_x_e(lr_s, ul_s)
with torch.no_grad():
    samps = model.batch_sample(dataset=test_ds, batch=hr_s.to(DEVICE), x_e=x_e,
                               sampler='euler', n_steps=FM_N_STEPS)
pred_phys = test_ds.unscale_data(samps[-1].cpu().numpy()[0], input_type='hr')
hr_phys   = test_ds.unscale_data(as_numpy(hr_s[0]),          input_type='hr')
lr_phys   = test_ds.unscale_data(as_numpy(lr_s[0]),          input_type='lr')
up_phys   = test_ds.unscale_data(as_numpy(ul_s[0]),          input_type='upscaled_lr')
enc_phys  = test_ds.unscale_data(as_numpy(model.lr_enc(lr_s.to(DEVICE).float()).cpu()[0]), input_type='hr')

fig, axes = plt.subplots(1, 5, figsize=(22, 4), dpi=120)
for ax, (ttl, data, phys) in zip(axes, [
        ('LR',     lr_phys[VIZ_CH],   lr_phys),
        ('Upscaled',up_phys[VIZ_CH],  up_phys),
        ('CNN',    enc_phys[VIZ_CH],  enc_phys),
        ('FM',     pred_phys[VIZ_CH], pred_phys),
        ('GT',     hr_phys[VIZ_CH],   hr_phys)]):
    ax.imshow(data.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
    _mk_contour(ax, phys[VIZ_CH:VIZ_CH+len(FIELD_NAMES)])
    ax.set_title(ttl, fontsize=9); ax.axis('off')
plt.suptitle(f'{RUN_NAME} — best pool sample (area={_best_area:.0f}px²)', fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
import os, wandb
api = wandb.Api()
_entity = os.environ.get("WANDB_ENTITY", "ngng-")

fig, ax = plt.subplots(figsize=(9, 4), dpi=130)
plotted = False

# ── Encoder curves (project: RRDN_Encoder) ───────────────────────────────────
enc_wb_name = '1_Sep_2026_encoder_temp'
enc_runs = list(api.runs(f"{_entity}/RRDN_Encoder",
                          filters={"display_name": enc_wb_name}))
if enc_runs:
    for col, ls in [("train_loss", "-"), ("test_loss", "--")]:
        _h = enc_runs[0].history(keys=[col], samples=10000, pandas=True)
        if col in _h.columns:
            vals = _h[col].dropna()
            print(f'Enc {col}: {len(vals)} epochs found')
            ax.plot(vals.values, ls=ls, label=f'Enc:{col}', lw=1.5, alpha=0.9)
            plotted = True
else:
    print(f'W&B: no run "{enc_wb_name}" in {_entity}/RRDN_Encoder')

# ── Flow matching curves (project: Flow3D_SuperResolution) ────────────────────
fm_runs = list(api.runs(f"{_entity}/Flow3D_SuperResolution",
                         filters={"display_name": WANDB_RUN_NAME}))
if fm_runs:
    for col, ls in [("train_loss", "-"), ("val_loss", "--")]:
        _h = fm_runs[0].history(keys=[col], samples=10000, pandas=True)
        if col in _h.columns:
            vals = _h[col].dropna()
            print(f'FM {col}: {len(vals)} epochs found')
            ax.plot(vals.values, ls=ls, label=col, lw=1.5, alpha=0.9)
            plotted = True
else:
    print(f'W&B: no run "{WANDB_RUN_NAME}" in {_entity}/Flow3D_SuperResolution')

# ── Fallback: local loss files ────────────────────────────────────────────────
if not plotted:
    from diffusionsr.runners.plot_training_curves import collect_curves
    all_c = [('Enc:'+l, v) for l,v in collect_curves(Path(ENC_RUN_DIR))] + collect_curves(Path(FLOW_RUN_DIR))
    for lbl, vals in all_c:
        ls = '--' if 'val' in lbl.lower() else '-'
        ax.plot(vals, ls=ls, label=lbl, lw=1.5, alpha=0.9)
        plotted = True
    if plotted: print(f'(fallback: local loss files)')

if plotted:
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'{RUN_NAME} — Training Curves (W&B)')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print('No loss data found in W&B or local files.')

In [ ]:
STATS_CH = (N_STEPS - 1) * len(FIELD_NAMES)  # current-timestep temperature channel

test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
maes, rmses, mp_errs, kh_errs, vc_maes = [], [], [], [], []
iou_list, cham_list, iou_gt_list, cham_gt_list = [], [], [], []
# physical meltpool metrics (µm / µm²)
mp_depth_pred, mp_depth_gt   = [], []
mp_length_pred, mp_length_gt = [], []
mp_area_pred, mp_area_gt     = [], []
kh_depth_pred, kh_depth_gt   = [], []
lwa_pred, lwa_gt             = [], []

t0 = time.perf_counter(); n_samples = 0
for i, batch in enumerate(test_loader):
    if ANALYSIS_MAX_BATCH is not None and i >= ANALYSIS_MAX_BATCH: break
    _, hr_b, lr_b, ul_b = batch[:4]
    xe = model.compute_x_e(lr_b, ul_b)
    with torch.no_grad():
        samps = model.batch_sample(dataset=test_ds, batch=hr_b.to(DEVICE),
                                   x_e=xe, sampler='euler', n_steps=FM_N_STEPS)
    pred_last = samps[-1].cpu(); del samps; torch.cuda.empty_cache()
    n_samples += hr_b.shape[0]
    for s in range(hr_b.shape[0]):
        p = test_ds.unscale_data(pred_last.numpy()[s], input_type='hr')
        g = test_ds.unscale_data(as_numpy(hr_b[s]),    input_type='hr')
        p_curr = p[STATS_CH:STATS_CH+len(FIELD_NAMES)]
        g_curr = g[STATS_CH:STATS_CH+len(FIELD_NAMES)]
        m = mae_rmse(p_curr[0], g_curr[0]); maes.append(m['MAE']); rmses.append(m['RMSE'])
        try:
            pmp, pkh = get_profile(p_curr[0:1]); gmp, gkh = get_profile(g_curr[0:1])
            mp_errs.append(float(np.mean(np.abs(pmp-gmp)))); kh_errs.append(float(np.mean(np.abs(pkh-gkh))))
        except: pass
        vc_maes.append(float(np.mean(np.abs((p_curr[0]>MELT_THRESHOLD).astype(float)-(g_curr[0]>MELT_THRESHOLD).astype(float)))))
        if has_T and has_liq:
            iou, _, cham  = consistency_metrics(p_curr[0]>T_LIQ, _liq_mask(p_curr))
            ioug,_, chamg = consistency_metrics(g_curr[0]>T_LIQ, _liq_mask(g_curr))
            iou_list.append(iou); cham_list.append(cham); iou_gt_list.append(ioug); cham_gt_list.append(chamg)
        # ── meltpool physical metrics ─────────────────────────────────────────
        try:
            mp_r = metrics_engine(p, metrics=_MP_METRICS_T, basis='temperature')
            mp_g = metrics_engine(g, metrics=_MP_METRICS_T, basis='temperature')
            mp_depth_pred.append(float(mp_r['depth_um']));   mp_depth_gt.append(float(mp_g['depth_um']))
            mp_length_pred.append(float(mp_r['length_um'])); mp_length_gt.append(float(mp_g['length_um']))
            mp_area_pred.append(float(mp_r['area_um2']));    mp_area_gt.append(float(mp_g['area_um2']))
        except: pass
        if _MP_METRICS_L:
            try:
                kh_r = metrics_engine(p, metrics=_MP_METRICS_L, basis='liquid')
                kh_g = metrics_engine(g, metrics=_MP_METRICS_L, basis='liquid')
                kh_depth_pred.append(float(kh_r['keyhole_depth_um']))
                kh_depth_gt.append(float(kh_g['keyhole_depth_um']))
                if kh_r['leading_wall_angle_valid']: lwa_pred.append(float(kh_r['leading_wall_angle_deg']))
                if kh_g['leading_wall_angle_valid']: lwa_gt.append(float(kh_g['leading_wall_angle_deg']))
            except: pass

elapsed = time.perf_counter() - t0
print(f'n={len(maes)} Euler n_steps={FM_N_STEPS}  MAE={np.nanmean(maes):.4f}±{np.nanstd(maes):.4f}')
print(f'  RMSE={np.nanmean(rmses):.4f}±{np.nanstd(rmses):.4f}')
if mp_errs: print(f'  MP-MAE={np.nanmean(mp_errs):.2f}±{np.nanstd(mp_errs):.2f}px  KH-MAE={np.nanmean(kh_errs):.2f}px')
if vc_maes: print(f'  VC-MAE={np.nanmean(vc_maes):.4f}±{np.nanstd(vc_maes):.4f}')
if iou_list:
    print(f'  IOU  pred={np.nanmean(iou_list):.4f}  GT={np.nanmean(iou_gt_list):.4f}')
    print(f'  Cham pred={np.nanmean(cham_list):.2f} px  GT={np.nanmean(cham_gt_list):.2f} px')
if mp_depth_pred:
    print(f'  [phys] depth_um   pred={np.nanmean(mp_depth_pred):.1f}±{np.nanstd(mp_depth_pred):.1f}  GT={np.nanmean(mp_depth_gt):.1f}±{np.nanstd(mp_depth_gt):.1f}')
    print(f'  [phys] length_um  pred={np.nanmean(mp_length_pred):.1f}±{np.nanstd(mp_length_pred):.1f}  GT={np.nanmean(mp_length_gt):.1f}±{np.nanstd(mp_length_gt):.1f}')
    print(f'  [phys] area_um2   pred={np.nanmean(mp_area_pred):.0f}±{np.nanstd(mp_area_pred):.0f}  GT={np.nanmean(mp_area_gt):.0f}±{np.nanstd(mp_area_gt):.0f}')
if kh_depth_pred:
    print(f'  [phys] kh_depth_um  pred={np.nanmean(kh_depth_pred):.1f}  GT={np.nanmean(kh_depth_gt):.1f}')
if lwa_pred:
    print(f'  [phys] lwa_deg    pred={np.nanmean(lwa_pred):.1f} (n={len(lwa_pred)})  GT={np.nanmean(lwa_gt):.1f} (n={len(lwa_gt)})')
print(f'  Speed:{n_samples/elapsed:.2f} s/s  {elapsed:.1f}s total')

In [ ]:
# ── Multi-sample grid ────────────────────────────────────────────────────────
VIZ_CH = (N_STEPS - 1) * len(FIELD_NAMES)
STATS_CH = VIZ_CH
N_GRID = 5

# Find best grid batch: max melt pool area from middle 50% of test set
_nb_g = max(1, len(test_ds) // N_GRID)
_step_g = max(1, _nb_g // 100)
_start_g, _end_g = _nb_g // 4, 3 * _nb_g // 4
_gl = DataLoader(test_ds, batch_size=N_GRID, shuffle=False, drop_last=True)
_gb = None; _best_area_g = -1
for _gi, _b in enumerate(_gl):
    if _gi > _end_g: break
    if _gi >= _start_g and (_gi - _start_g) % _step_g == 0:
        _, _hr_b, _, _ = _b[:4]
        _a = float((test_ds.unscale_data(_hr_b.numpy()[0], input_type='hr')[VIZ_CH] > MELT_THRESHOLD).sum())
        if _a > _best_area_g: _best_area_g, _gb = _a, _b
if _gb is None:
    for _gi, _gb in enumerate(DataLoader(test_ds, batch_size=N_GRID, shuffle=False, drop_last=True)):
        if _gi == _nb_g // 2: break
_res_g, _hr_g, _lr_g, _ul_g = _gb[:4]; _xe_g = model.compute_x_e(_lr_g, _ul_g)
with torch.no_grad():
    _sg = model.batch_sample(dataset=test_ds, batch=_hr_g.to(DEVICE), x_e=_xe_g,
                             sampler='euler', n_steps=FM_N_STEPS)
    _ec_g = model.lr_enc(_lr_g.to(DEVICE).float()).cpu()
_COLS = ['LR Input','Upscaled LR','CNN Encoder','FlowMatching','GT']
fig, axes = plt.subplots(N_GRID, 5, figsize=(22, 3.5*N_GRID), dpi=90)
if N_GRID == 1: axes = axes[np.newaxis]
for r in range(N_GRID):
    _p = test_ds.unscale_data(_sg[-1].cpu().numpy()[r], input_type='hr')
    _g = test_ds.unscale_data(as_numpy(_hr_g[r]),       input_type='hr')
    _l = test_ds.unscale_data(as_numpy(_lr_g[r]),       input_type='lr')
    _u = test_ds.unscale_data(as_numpy(_ul_g[r]),       input_type='upscaled_lr')
    _e = test_ds.unscale_data(as_numpy(_ec_g[r]),       input_type='hr')
    for c, (data, phys) in enumerate(zip(
            [_l[VIZ_CH], _u[VIZ_CH], _e[VIZ_CH], _p[VIZ_CH], _g[VIZ_CH]],
            [_l, _u, _e, _p, _g])):
        ax = axes[r, c]
        ax.imshow(data.T, origin='lower', cmap='jet', vmin=293, vmax=5000, aspect='auto')
        _mk_contour(ax, phys[VIZ_CH:VIZ_CH+len(FIELD_NAMES)]); ax.axis('off')
        if r == 0: ax.set_title(_COLS[c], fontsize=9, fontweight='bold')
plt.suptitle(f'{RUN_NAME} — multi-sample grid', fontsize=10); plt.tight_layout(); plt.show()

# ── Euler step ablation ───────────────────────────────────────────────────────
_ABL_NSTEPS=[5,10,25,50,100]; _ABL_N=4; _ab_mae={n:[] for n in _ABL_NSTEPS}; _ab_mp={n:[] for n in _ABL_NSTEPS}
for _ai,_ab in enumerate(DataLoader(test_ds,batch_size=BATCH_SIZE,shuffle=False)):
    if _ai>=_ABL_N: break
    _,_ahr,_alr,_aul=_ab[:4]; _axe=model.compute_x_e(_alr,_aul)
    for _ns in _ABL_NSTEPS:
        with torch.no_grad(): _as=model.batch_sample(dataset=test_ds,batch=_ahr.to(DEVICE),x_e=_axe,sampler='euler',n_steps=_ns)
        for _s in range(_ahr.shape[0]):
            _pp=test_ds.unscale_data(_as[-1].cpu().numpy()[_s],input_type='hr')
            _gg=test_ds.unscale_data(as_numpy(_ahr[_s]),input_type='hr')
            _ab_mae[_ns].append(mae_rmse(_pp[STATS_CH],_gg[STATS_CH])['MAE'])
            try: _pmp,_=get_profile(_pp[STATS_CH:STATS_CH+1]); _gmp,_=get_profile(_gg[STATS_CH:STATS_CH+1]); _ab_mp[_ns].append(float(np.mean(np.abs(_pmp-_gmp))))
            except: pass
_mae_v=[np.nanmean(_ab_mae[n]) for n in _ABL_NSTEPS]; _mp_v=[np.nanmean(_ab_mp[n]) if _ab_mp[n] else float('nan') for n in _ABL_NSTEPS]
fig,(ax1,ax2)=plt.subplots(1,2,figsize=(12,5),dpi=120)
ax1.plot(_ABL_NSTEPS,_mae_v,'o-',color='darkorange',lw=2,ms=8); ax1.set_xlabel('Euler steps'); ax1.set_ylabel('MAE'); ax1.set_title('Euler Ablation'); ax1.grid(True,alpha=0.3)
ax2.plot(_ABL_NSTEPS,_mp_v,'s-',color='green',lw=2,ms=8); ax2.set_xlabel('Euler steps'); ax2.set_ylabel('MP-MAE (px)'); ax2.set_title('MP Depth'); ax2.grid(True,alpha=0.3)
plt.suptitle(f'{RUN_NAME} — step ablation',fontsize=11); plt.tight_layout(); plt.show()

In [ ]:
import os; os.makedirs(EVAL_OUT_DIR, exist_ok=True)
summary = {
    'run_name': RUN_NAME, 'wandb_run': WANDB_RUN_NAME, 'model': 'FlowMatching',
    'encoding': ENCODING, 'conditioning': CONDITIONING, 'fields': str(FIELD_NAMES),
    'fm_n_steps': FM_N_STEPS, 'n_test': len(maes),
    'mae_mean': float(np.nanmean(maes)),   'mae_std': float(np.nanstd(maes)),
    'rmse_mean': float(np.nanmean(rmses)), 'rmse_std': float(np.nanstd(rmses)),
    'mp_mae_mean': float(np.nanmean(mp_errs)) if mp_errs else float('nan'),
    'kh_mae_mean': float(np.nanmean(kh_errs)) if kh_errs else float('nan'),
    'vc_mae_mean': float(np.nanmean(vc_maes)) if vc_maes else float('nan'),
    'iou_mean':    float(np.nanmean(iou_list)) if iou_list else float('nan'),
    'depth_um_pred':    float(np.nanmean(mp_depth_pred))  if mp_depth_pred  else float('nan'),
    'depth_um_gt':      float(np.nanmean(mp_depth_gt))    if mp_depth_gt    else float('nan'),
    'length_um_pred':   float(np.nanmean(mp_length_pred)) if mp_length_pred else float('nan'),
    'length_um_gt':     float(np.nanmean(mp_length_gt))   if mp_length_gt   else float('nan'),
    'area_um2_pred':    float(np.nanmean(mp_area_pred))   if mp_area_pred   else float('nan'),
    'area_um2_gt':      float(np.nanmean(mp_area_gt))     if mp_area_gt     else float('nan'),
    'kh_depth_um_pred': float(np.nanmean(kh_depth_pred))  if kh_depth_pred  else float('nan'),
    'kh_depth_um_gt':   float(np.nanmean(kh_depth_gt))    if kh_depth_gt    else float('nan'),
    'lwa_deg_pred':     float(np.nanmean(lwa_pred))        if lwa_pred       else float('nan'),
    'lwa_deg_gt':       float(np.nanmean(lwa_gt))          if lwa_gt         else float('nan'),
}
pd.DataFrame([summary]).to_csv(f'{EVAL_OUT_DIR}/metrics_summary.csv', index=False)
print(f'Saved -> {EVAL_OUT_DIR}/metrics_summary.csv'); pd.DataFrame([summary]).T